# Convolution of Periodic Splines
We select the <span style="color:#8c564b">**sepia**</span> color to plot the spline $h$ that acts as smoothing filter. Moreover, we select the <span style="color:#e0e0e0">**light gray**</span> color to plot the spline $f$ being filtered. The result $h*f$ of their convolution is shown as a <span style="color:#1f77b4">**blue**</span> curve, with data samples at the integers represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline with absolute Cauchy coefficients
h = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_cauchy(6), degree = 3)
h.spline_coeff = np.abs(h.spline_coeff)
h = h.times(1.0 / np.sum(h.spline_coeff))
# Initial random periodic linear spline with normal Gaussian coefficients
f = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 1)

# Plot
def update_plot (
    period = 6,
    sep1 = "",
    degree_h = 3,
    delay_h = 0.0,
    sep2 = "",
    degree_f = 1,
    delay_f = 0.0
):
    global h
    global f

    # Update of the Cauchy spline
    if h.period != period:
        h = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_cauchy(period),
            degree = h.degree
        )
        h.spline_coeff = np.abs(h.spline_coeff)
        h = h.times(1.0 / np.sum(h.spline_coeff))
    h.degree = degree_h
    h.delay = delay_h
    # Update of the Gaussian spline
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = f.degree
        )
    f.degree = degree_f
    f.delay = delay_f

    # Convolution
    g = sk.PeriodicSpline1D.convolve(h, f)

    # Dynamic range
    image = {h.image(), f.image(), g.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plot of the splines
    subplot = plt.subplots()
    h.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "-C5",
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " "
    )
    f.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "#e0e0e0",
        curve_lw = 3.0,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " "
    )
    g.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        knot_marker = " "
    )
    plt.show()

# Interaction
widgets.interactive(
    update_plot,
    period = (1, max_period),
    sep1 = widgets.HTML(
        value="<hr style='border:1px solid;margin:15px 0;width:135px'>",
        description = "• • • • •"
    ),
    degree_h = (0, max_degree),
    delay_h = (-max_delay, max_delay),
    sep2 = widgets.HTML(
        value="<hr style='border:1px solid;margin:15px 0;width:135px'>",
        description = "• • • • •"
    ),
    degree_f = (0, max_degree),
    delay_f = (-max_delay, max_delay)
)
